# Graph + FGW with human-labeled examples

Each tagged transcript is a graph (one node per cue, plus the start node). One FGW barycenter per level is computed from its labeled examples (as many nodes as the median of their graphs), and every transcript gets the level of the nearest barycenter.

**Data**: a csv with one row per transcript, its tagged text (`Transcript`) and the level given by humans (`ability_label_name`: Low / Medium / High).

**Protocol** (identical in `human_fgw.ipynb` and `human_embeddings.ipynb`):
* stratified $k$-fold cross-validation on the human labels;
* for each fold, $m$ labeled transcripts per level are drawn from the other folds (all of them with `M = None`), and the transcripts of the fold are classified with them;
* the folds and examples are saved in `splits/` and reloaded, so both methods use exactly the same ones;
* scores against the human level: accuracy, macro-F1, quadratic-weighted Cohen's $\kappa$ with a 95% bootstrap CI over transcripts;
* robustness: the whole cross-validation is repeated with `N_REPEATS` different random splits (mean ± SD).

In [ ]:
import os
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

sys.path.append(os.path.realpath("../lib"))
from evaluation import (LEVELS, evaluate, load_labeled_transcripts, load_or_create_split, plot_confusion,
                        scores, summarize)
from graph_multimodalsentence import Graph, compute_barycenter
from ot_distances import Fused_Gromov_Wasserstein_distance

## Parameters

In [ ]:
LABELS_FILE = "human_labels.csv"
TEXT_COLUMN = "Transcript"
LABEL_COLUMN = "ability_label_name"
DIMENSION = "ability"           # names the splits and outputs

SPLIT_DIR = "splits"            # shared with the other notebook
OUTPUT_DIR = "results"

M = 10                          # labeled examples per level (None: all the transcripts of the other folds)
N_FOLDS = 5
SEED = 0                        # split of the main results
N_REPEATS = 5                   # random splits for the robustness check (seeds SEED, SEED + 1, ...)

EMBEDDING_MODEL = "sentence-transformers/bert-base-nli-mean-tokens"
ALPHA = 0.14                    # set to the value selected in tune_alpha.ipynb
BARYCENTER_MAX_ITER = 50

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Human-labeled transcripts

In [ ]:
df = load_labeled_transcripts(LABELS_FILE, TEXT_COLUMN, LABEL_COLUMN)
counts = df["level"].value_counts().reindex(LEVELS, fill_value=0)
print(f"{len(df)} transcripts")
if M is not None:
    needed = int(np.ceil(M * N_FOLDS / (N_FOLDS - 1)))
    print(f"m = {M} examples per level from {N_FOLDS - 1} folds out of {N_FOLDS}: "
          f"about {needed} transcripts per level are needed")
counts.to_frame("transcripts")

## Graphs

Cue descriptions are embedded once (the same descriptions occur in many transcripts).

In [ ]:
class CachedEncoder:
    # Wraps a SentenceTransformer so that each description is encoded only once.
    def __init__(self, model):
        self.model = model
        self.cache = {}

    def encode(self, text):
        if text not in self.cache:
            self.cache[text] = self.model.encode(text)
        return self.cache[text]


embedding = CachedEncoder(SentenceTransformer(EMBEDDING_MODEL))
graphs = {i: Graph.from_sentence(text, embedding) for i, text in zip(df["id"], df["text"])}
fgw = Fused_Gromov_Wasserstein_distance(alpha=ALPHA, features_metric="sqeuclidean", method="temporal")

## Nearest FGW barycenter

In [ ]:
def classify(folds, examples):
    # Predicted level of every transcript, from the barycenters of its own fold.
    predictions = []
    for fold in sorted(folds["fold"].unique()):
        test = folds[folds["fold"] == fold]
        support = examples[examples["fold"] == fold]
        barycenters = [compute_barycenter([graphs[i] for i in support.loc[support["level"] == level, "id"]],
                                          alpha=ALPHA, max_iter=BARYCENTER_MAX_ITER) for level in LEVELS]
        distances = np.array([[fgw.graph_d(graphs[i], b) for b in barycenters] for i in test["id"]])
        pred = pd.DataFrame({"id": test["id"], "fold": fold, "true": test["level"],
                             "pred": np.array(LEVELS)[distances.argmin(axis=1)]})
        for k, level in enumerate(LEVELS):
            pred[f"distance_{level}"] = distances[:, k]
        predictions.append(pred)
    return pd.concat(predictions, ignore_index=True)

## Classification (main split)

In [ ]:
SETTING = f"m = {M}" if M is not None else "all examples"
folds, examples = load_or_create_split(df, DIMENSION, f"human_seed{SEED}", SPLIT_DIR, M, N_FOLDS, SEED)
pred = classify(folds, examples)
pred.to_csv(os.path.join(OUTPUT_DIR, f"human_fgw_predictions_{DIMENSION}.csv"), index=False)

results = evaluate({(DIMENSION, SETTING): pred}, method="graph + FGW", reference="human")
results.to_csv(os.path.join(OUTPUT_DIR, f"human_fgw_results_{DIMENSION}.csv"), index=False)
summarize(results)

All folds pooled, with the 95% bootstrap CI of $\kappa$:

In [ ]:
pooled = results[results["fold"] == "all"].drop(columns="fold").copy()
pooled["kappa [95% CI]"] = [f"{k:.2f} [{lo:.2f}, {hi:.2f}]" for k, lo, hi in
                            zip(pooled["kappa"], pooled["kappa_ci_low"], pooled["kappa_ci_high"])]
pooled.round(3)

In [ ]:
plot_confusion({(DIMENSION, SETTING): pred}, "human", os.path.join(OUTPUT_DIR, f"human_fgw_confusion_{DIMENSION}.png"))
plt.show()

## Robustness over random splits

The same cross-validation with `N_REPEATS` different splits (the folds and examples of each are saved too).

In [ ]:
repeats = []
for seed in range(SEED, SEED + N_REPEATS):
    folds_r, examples_r = load_or_create_split(df, DIMENSION, f"human_seed{seed}", SPLIT_DIR, M, N_FOLDS, seed)
    pred_r = pred if seed == SEED else classify(folds_r, examples_r)
    repeats.append({"seed": seed, **scores(pred_r)})
repeats = pd.DataFrame(repeats)
repeats.to_csv(os.path.join(OUTPUT_DIR, f"human_fgw_repeats_{DIMENSION}.csv"), index=False)
display(repeats.round(3))
repeats[["accuracy", "macro_f1", "kappa"]].agg(["mean", "std"]).round(3)

## Comparison with the embedding baseline

If `human_embeddings.ipynb` was run with the same parameters (same splits).

In [ ]:
paths = {"embeddings": os.path.join(OUTPUT_DIR, f"human_embedding_results_{DIMENSION}.csv"),
         "repeats": os.path.join(OUTPUT_DIR, f"human_embedding_repeats_{DIMENSION}.csv")}
if os.path.exists(paths["embeddings"]):
    both = pd.concat([results, pd.read_csv(paths["embeddings"])])
    both = both[both["fold"].astype(str) == "all"]
    display(both.pivot_table(index="setting", columns="method", values=["accuracy", "macro_f1", "kappa"]).round(3))
if os.path.exists(paths["repeats"]):
    display(pd.concat({"graph + FGW": repeats, "embeddings": pd.read_csv(paths["repeats"])})
            [["accuracy", "macro_f1", "kappa"]].groupby(level=0).agg(["mean", "std"]).round(3))